# Demo Modul 3: Optimizer dan Strategi Pelatihan

**Durasi sesi:** 120 menit  
**Kasus:** Fashion-MNIST, FNN $784 \rightarrow 128 \rightarrow 10$.

Modul 2 memastikan gradiennya benar. Notebook ini membahas pertanyaan berikutnya: bagaimana gradien itu sebaiknya diterjemahkan menjadi langkah pembaruan.

## Capaian demo

Setelah demo, praktikan dapat:

1. menyiapkan split dan fungsi pelatihan yang sama untuk seluruh run;
2. membaca gejala learning rate terlalu kecil dan terlalu besar dari kurva;
3. menjalankan perbandingan optimizer yang adil;
4. memilih model dari validation set, bukan dari data uji; dan
5. menilai apakah scheduler dan perubahan batch size benar-benar menolong.

In [ ]:
import platform
import random
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from torchvision import transforms
from torchvision.datasets import FashionMNIST

SEED = 42
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(SEED)
pd.set_option('display.precision', 4)
print({'python': platform.python_version(), 'torch': torch.__version__,
       'device': str(DEVICE)})

## 1. Data: subset tetap dan normalisasi dari train

Protokol modul: $12\,000$ citra latih dan $3\,000$ citra validasi diambil terstratifikasi dari data latih resmi. Statistik normalisasi dihitung **hanya** dari subset latih.

In [ ]:
DATA_ROOT = Path('../../data/raw')
if not DATA_ROOT.exists():
    DATA_ROOT = Path('data/raw')

latih_penuh = FashionMNIST(root=DATA_ROOT, train=True, download=False,
                           transform=transforms.ToTensor())
uji_resmi = FashionMNIST(root=DATA_ROOT, train=False, download=False,
                         transform=transforms.ToTensor())

X_penuh = latih_penuh.data.float().unsqueeze(1) / 255.0    # (60000, 1, 28, 28)
y_penuh = latih_penuh.targets
X_uji = uji_resmi.data.float().unsqueeze(1) / 255.0
y_uji = uji_resmi.targets

idx_latih, idx_val = train_test_split(
    np.arange(len(y_penuh)), train_size=12_000, test_size=3_000,
    stratify=y_penuh.numpy(), random_state=SEED)

X_latih, y_latih = X_penuh[idx_latih], y_penuh[idx_latih]
X_val, y_val = X_penuh[idx_val], y_penuh[idx_val]

MEAN, STD = X_latih.mean().item(), X_latih.std().item()   # dari subset latih saja
normalkan = lambda t: (t - MEAN) / STD

ds_latih = TensorDataset(normalkan(X_latih), y_latih)
ds_val = TensorDataset(normalkan(X_val), y_val)
ds_uji = TensorDataset(normalkan(X_uji), y_uji)

print(f'latih {len(ds_latih)}  validasi {len(ds_val)}  uji {len(ds_uji)}')
print(f'mean {MEAN:.4f}  std {STD:.4f}')
print('distribusi kelas latih:', torch.bincount(y_latih).tolist())

**Pemeriksaan:** ketiga split berukuran $12\,000$, $3\,000$, dan $10\,000$; setiap kelas pada subset latih berisi $1\,200$ citra karena pengambilannya terstratifikasi.

## 2. Satu model, satu fungsi pelatihan

Seluruh run memakai fungsi yang sama. Menyalin ulang kode pelatihan untuk tiap run adalah sumber ketidakadilan yang paling sering terjadi.

In [ ]:
BATCH = 128
EPOCH = 5

def buat_model():
    seed_everything(SEED)          # inisialisasi identik untuk setiap run
    return nn.Sequential(
        nn.Flatten(),
        nn.Linear(28 * 28, 128),
        nn.ReLU(),
        nn.Linear(128, 10),
    ).to(DEVICE)

def buat_optimizer(nama, params, lr):
    if nama == 'sgd':
        return torch.optim.SGD(params, lr=lr)
    if nama == 'momentum':
        return torch.optim.SGD(params, lr=lr, momentum=0.9)
    if nama == 'adam':
        return torch.optim.Adam(params, lr=lr)
    raise ValueError(nama)

def loader(ds, batch, acak):
    g = torch.Generator().manual_seed(SEED)
    return DataLoader(ds, batch_size=batch, shuffle=acak, generator=g if acak else None)

@torch.no_grad()
def evaluasi(model, ds, batch=512):
    model.eval()
    kriteria = nn.CrossEntropyLoss(reduction='sum')
    total_loss, benar = 0.0, 0
    for xb, yb in loader(ds, batch, False):
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        logits = model(xb)
        total_loss += kriteria(logits, yb).item()
        benar += (logits.argmax(1) == yb).sum().item()
    return total_loss / len(ds), benar / len(ds)

print('jumlah parameter:', sum(p.numel() for p in buat_model().parameters()))

**Pemeriksaan:** jumlah parameter harus $101\,770$, yaitu $784\cdot128+128$ ditambah $128\cdot10+10$.

In [ ]:
def jalankan(nama_opt, lr, batch=BATCH, epoch=EPOCH, scheduler=None):
    model = buat_model()
    opt = buat_optimizer(nama_opt, model.parameters(), lr)
    sched = None
    if scheduler == 'step':
        sched = torch.optim.lr_scheduler.StepLR(opt, step_size=2, gamma=0.5)
    elif scheduler == 'cosine':
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epoch)

    kriteria = nn.CrossEntropyLoss()
    dl = loader(ds_latih, batch, True)
    riwayat = {'train_loss': [], 'val_loss': [], 'val_acc': []}
    norma, n_update, mulai = [], 0, time.perf_counter()

    for _ in range(epoch):
        model.train()
        jumlah = 0.0
        for xb, yb in dl:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            loss = kriteria(model(xb), yb)
            loss.backward()
            norma.append(torch.sqrt(sum((p.grad ** 2).sum() for p in model.parameters())).item())
            opt.step()
            jumlah += loss.item() * len(yb)
            n_update += 1
        if sched is not None:
            sched.step()
        vl, va = evaluasi(model, ds_val)
        riwayat['train_loss'].append(jumlah / len(ds_latih))
        riwayat['val_loss'].append(vl)
        riwayat['val_acc'].append(va)

    return model, riwayat, {
        'run_id': f'{nama_opt}-lr{lr}-b{batch}-{scheduler or "none"}',
        'seed': SEED, 'optimizer': nama_opt, 'learning_rate': lr,
        'scheduler': scheduler or 'none', 'batch_size': batch,
        'n_update': n_update, 'train_loss': riwayat['train_loss'][-1],
        'val_loss': riwayat['val_loss'][-1], 'val_acc': riwayat['val_acc'][-1],
        'grad_norm_mean': float(np.mean(norma)),
        'runtime_s': time.perf_counter() - mulai,
    }

## 3. Baseline dan tiga learning rate

Optimizer yang sama, tiga learning rate. Perhatikan bentuk kurvanya, bukan hanya angka akhirnya.

In [ ]:
baris = []
kurva = {}
for lr in (0.001, 0.1, 1.0):
    _, riwayat, catatan = jalankan('sgd', lr)
    kurva[f'SGD lr={lr}'] = riwayat
    baris.append(catatan)

pd.DataFrame(baris)[['run_id', 'train_loss', 'val_loss', 'val_acc',
                     'grad_norm_mean', 'runtime_s']]

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.4))
for label, r in kurva.items():
    ax.plot(range(1, EPOCH + 1), r['val_loss'], marker='o', label=label)
ax.set_xlabel('epoch'); ax.set_ylabel('validation loss')
ax.set_title('Pengaruh learning rate pada SGD'); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

**Bacaan gejala.** $\eta=0{,}001$ menghasilkan kurva yang hampir lurus dan turun sangat lambat; $\eta=1{,}0$ menghasilkan kurva bergerigi atau meningkat. Nilai di antaranya turun cepat lalu melandai. Bandingkan pula kolom `grad_norm_mean`: learning rate yang terlalu besar biasanya disertai norma gradien yang jauh lebih besar.

## 4. Sembilan run terkendali

Setiap optimizer diberi kesempatan pada learning rate-nya sendiri. Inilah yang membuat perbandingan menjadi adil.

In [ ]:
grid = {'sgd': [0.01, 0.1, 0.5],
        'momentum': [0.01, 0.05, 0.1],
        'adam': [1e-4, 1e-3, 1e-2]}

hasil, riwayat_semua = [], {}
for nama_opt, daftar_lr in grid.items():
    for lr in daftar_lr:
        _, riwayat, catatan = jalankan(nama_opt, lr)
        hasil.append(catatan)
        riwayat_semua[catatan['run_id']] = riwayat

tabel = pd.DataFrame(hasil)
tabel[['run_id', 'optimizer', 'learning_rate', 'val_loss', 'val_acc',
       'runtime_s']].sort_values('val_loss')

## 5. Memilih pemenang dari validation loss

In [ ]:
pemenang = tabel.loc[tabel.groupby('optimizer')['val_loss'].idxmin()]
print(pemenang[['optimizer', 'learning_rate', 'val_loss', 'val_acc', 'runtime_s']]
      .to_string(index=False))

fig, ax = plt.subplots(figsize=(6, 3.4))
for _, r in pemenang.iterrows():
    ax.plot(range(1, EPOCH + 1), riwayat_semua[r['run_id']]['val_loss'],
            marker='o', label=f"{r['optimizer']} (lr={r['learning_rate']})")
ax.set_xlabel('epoch'); ax.set_ylabel('validation loss')
ax.set_title('Tiga optimizer pada learning rate terbaiknya'); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

Perhatikan kolom `runtime_s`: optimizer dengan validation loss terendah belum tentu yang tercepat. Kedua angka itu harus dilaporkan bersama.

## 6. Scheduler dan batch size

Scheduler dinilai pada anggaran epoch yang sama. Batch size mengubah jumlah update **dan** derau gradien sekaligus, sehingga pengaruhnya tidak boleh dibaca sebagai satu faktor tunggal.

In [ ]:
terbaik = pemenang.sort_values('val_loss').iloc[0]
opt_terbaik, lr_terbaik = terbaik['optimizer'], terbaik['learning_rate']
print(f'konfigurasi terbaik: {opt_terbaik} lr={lr_terbaik}')

tambahan = []
for sch in (None, 'step', 'cosine'):
    _, _, catatan = jalankan(opt_terbaik, lr_terbaik, scheduler=sch)
    tambahan.append(catatan)
for b in (32, 128, 512):
    _, _, catatan = jalankan(opt_terbaik, lr_terbaik, batch=b)
    tambahan.append(catatan)

pd.DataFrame(tambahan)[['run_id', 'scheduler', 'batch_size', 'n_update',
                        'val_loss', 'val_acc', 'runtime_s']]

## Exit ticket

1. Mengapa learning rate terbaik Adam berada pada skala yang jauh lebih kecil daripada SGD?
2. Berapa jumlah update satu epoch pada batch $32$ dan pada batch $512$, dan mengapa berbeda?
3. Apa dua angka yang harus dilaporkan bersama sebelum menyatakan satu optimizer lebih baik?

**Tugas setelah sesi:** kerjakan `starter-mahasiswa.ipynb` — sembilan run inti, tiga run scheduler, tiga run batch size, lalu pilih satu optimizer dan evaluasi data uji satu kali.